In [1]:
# %pip install langgraph langchain-openai pydantic

In [ ]:
from typing import Any, Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict


class RiskReport(BaseModel):
    order_id: str = Field(description="The order ID extracted from ticket")
    user_id: str = Field(description="The user ID attached to the order")
    risk_score: int = Field(description="Fraud risk score 0-100 from audit_fraud_risk")
    risk_band: Literal["low", "medium", "high"] = Field(description="Risk category level")
    triggered_rules: list[dict[str, Any]] = Field(
        default_factory=list, description="Rules triggered during audit"
    )
    blocks_automatic_refund: bool = Field(description="True if risk_score > 70 or high risk band")


class Decision(BaseModel):
    order_id: str
    user_id: str
    verdict: Literal["ELIGIBLE", "INELIGIBLE", "ESCALATION_REQUIRED"] = Field(
        description="Return policy check result"
    )
    refund_status: Literal["APPROVED", "REJECTED", "ESCALATION_REQUIRED"] = Field(
        description="Final operational refund status"
    )
    approved_amount: float = Field(default=0.0, description="Refund amount approved")
    risk_band: str = Field(description="Carried over from RiskReport")
    rationale: str = Field(description="Internal business logic explaining the decision")


class FinalResolution(BaseModel):
    customer_response: str = Field(
        description="Polite message to customer. DO NOT disclose fraud flags."
    )
    slack_alert_sent: bool = Field(
        description="True if an alert was posted to outbox/alerts.jsonl"
    )
    escalation_channel: str | None = Field(default=None, description="Target channel if escalated")


class AgentState(TypedDict):
    ticket_text: str
    risk_report: RiskReport | None
    decision: Decision | None
    final_resolution: FinalResolution | None

In [3]:
import multi_agent_tools as mat

researcher_tools = mat.RESEARCHER_TOOLS
decision_tools = mat.DECISION_TOOLS
comms_tools = mat.COMMS_TOOLS

In [ ]:
import time

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:
def researcher_node(state: AgentState) -> dict:
    ticket = state["ticket_text"]

    class ExtractedTicket(BaseModel):
        order_id: str

    extractor = llm.with_structured_output(ExtractedTicket)
    extracted = extractor.invoke(
        f"Extract the order_id (e.g. ORD-1001) from this ticket: {ticket}"
    )
    order_id = extracted.order_id

    order_details = mat.get_order_details(order_id)

    if "error" in order_details:
        risk_report = RiskReport(
            order_id=order_id,
            user_id="UNKNOWN",
            risk_score=100,
            risk_band="high",
            triggered_rules=[{"rule_id": "ORDER_NOT_FOUND"}],
            blocks_automatic_refund=True,
        )
        return {"risk_report": risk_report}

    user_id = order_details["user_id"]
    fraud_audit = mat.audit_fraud_risk(order_id, user_id)

    risk_score = fraud_audit.get("risk_score", 0)
    risk_band = fraud_audit.get("risk_band", "low")
    blocks = risk_score > 70 or risk_band == "high"

    risk_report = RiskReport(
        order_id=order_id,
        user_id=user_id,
        risk_score=risk_score,
        risk_band=risk_band,
        triggered_rules=fraud_audit.get("triggered_rules", []),
        blocks_automatic_refund=blocks,
    )

    time.sleep(1)
    return {"risk_report": risk_report}

In [6]:
def decision_node(state: AgentState) -> dict:
    risk_report = state["risk_report"]
    ticket = state["ticket_text"]

    class ReturnReason(BaseModel):
        reason: Literal[
            "damaged_on_arrival",
            "wrong_item",
            "item_missing",
            "late_delivery",
            "changed_mind",
        ]

    reason_extracted = (
        llm.with_structured_output(ReturnReason)
        .invoke(f"Identify the return reason from ticket: {ticket}")
        .reason
    )

    policy_res = mat.check_return_policy(risk_report.order_id, reason_extracted)
    verdict = policy_res.get("verdict", "INELIGIBLE")

    if risk_report.blocks_automatic_refund:
        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="ESCALATION_REQUIRED",
            approved_amount=0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Refund blocked due to fraud risk audit (Score: {risk_report.risk_score}). Policy was: {verdict}.",
        )
    elif verdict == "ELIGIBLE":
        order_details = mat.get_order_details(risk_report.order_id)
        amount = order_details.get("total_amount", 0.0)

        refund_res = mat.process_refund(risk_report.order_id, amount, reason_extracted)
        status = refund_res.get("status", "REJECTED")

        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="APPROVED" if status == "APPROVED" else status,
            approved_amount=amount if status == "APPROVED" else 0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Refund processed with status: {status}.",
        )
    else:
        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="REJECTED",
            approved_amount=0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Ineligible based on policy: {policy_res.get('details')}",
        )

    time.sleep(1)
    return {"decision": decision}

In [7]:
def comms_node(state: AgentState) -> dict:
    risk_report = state["risk_report"]
    decision = state["decision"]

    slack_sent = False
    escalation_channel = None

    if decision.refund_status == "ESCALATION_REQUIRED" or risk_report.blocks_automatic_refund:
        route_info = mat.get_escalation_route(
            risk_band=risk_report.risk_band,
            requested_amount=decision.approved_amount,
            verdict=decision.verdict,
        )

        if route_info.get("escalation_required"):
            channel_id = route_info["channel_id"]
            severity = route_info["severity"]
            escalation_channel = route_info["channel"]

            payload = {
                "order_id": decision.order_id,
                "user_id": decision.user_id,
                "risk_score": risk_report.risk_score,
                "risk_band": risk_report.risk_band,
                "rationale": decision.rationale,
            }
            alert_res = mat.send_slack_alert(
                channel_id=channel_id,
                severity=severity,
                payload=payload,
            )
            slack_sent = alert_res.get("delivered", False)

    prompt = f"""
    Formulate a polite, professional response to the customer for order {decision.order_id}.
    Decision Status: {decision.refund_status}.
    Approved Amount: ${decision.approved_amount}.

    STRICT GUARDRAIL:
    - DO NOT mention terms like "fraud", "risk score", "suspicious activity", or "audit".
    - If status is ESCALATION_REQUIRED, explain that the request requires manual verification by our specialized team and we will follow up shortly.
    """

    customer_msg = llm.invoke(prompt).content.strip()

    final_res = FinalResolution(
        customer_response=customer_msg,
        slack_alert_sent=slack_sent,
        escalation_channel=escalation_channel,
    )

    return {"final_resolution": final_res}

In [8]:
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("decision", decision_node)
workflow.add_node("comms", comms_node)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "decision")
workflow.add_edge("decision", "comms")
workflow.add_edge("comms", END)

app = workflow.compile()
ticket_1005 = "Hi, my order ORD-1005 arrived broken. I demand a full refund now."

print("--- Executing Multi-Agent Pipeline for ORD-1005 ---")
res = app.invoke({"ticket_text": ticket_1005})

print("\n[Risk Report]:", res["risk_report"].model_dump())
print("\n[Decision]:", res["decision"].model_dump())
print("\n[Final Resolution]:", res["final_resolution"].model_dump())

--- Executing Multi-Agent Pipeline for ORD-1005 ---

[Risk Report]: {'order_id': 'ORD-1005', 'user_id': 'USR-105', 'risk_score': 90, 'risk_band': 'high', 'triggered_rules': [{'rule_id': 'FR-01', 'name': 'repeat_refund_claims', 'weight': 25, 'why': '3 claims in the last 60 days'}, {'rule_id': 'FR-02', 'name': 'last_minute_address_change', 'weight': 20, 'why': 'address changed 2 day(s) before delivery'}, {'rule_id': 'FR-04', 'name': 'prior_fraud_flag', 'weight': 15, 'why': '1 prior fraud flag(s) on record'}, {'rule_id': 'FR-05', 'name': 'high_initial_fraud_score', 'weight': 15, 'why': 'inherited fraud score is 61'}, {'rule_id': 'FR-08', 'name': 'refund_amount_velocity', 'weight': 15, 'why': '451.00 USD refunded in the last 60 days'}], 'blocks_automatic_refund': True}

[Decision]: {'order_id': 'ORD-1005', 'user_id': 'USR-105', 'verdict': 'ELIGIBLE', 'refund_status': 'ESCALATION_REQUIRED', 'approved_amount': 0.0, 'risk_band': 'high', 'rationale': 'Refund blocked due to fraud risk audit (Sc